# Asahlagi — Fine-tune Quiz Generator (IndoT5)

**Owner**: Audry (Quiz Generator)

Fine-tunes `Wikidepia/IndoT5-base` for Indonesian question generation on TyDiQA-id,
then pushes the model to the Hugging Face Hub.

**Important**
- Set runtime to **GPU (T4)**: Runtime → Change runtime type → T4 GPU.
- `fp16` MUST stay **False** — T5 + fp16 caused NaN/garbage in the first attempt.
- The input prefix here (`"buat pertanyaan: "`) must match the inference side (HF Space).
- The model only learns *passage → question*. Answer options/distractors are produced
  separately by the backend (`app/services/_distractors.py`).

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==4.45.2" "datasets==3.0.1" sentencepiece accelerate evaluate sacrebleu

## 2. Prepare data (TyDiQA-id, inline)

Downloads TyDiQA Gold Passage, keeps the Indonesian subset, and builds
`(input, target)` pairs: input = `"buat pertanyaan: {answer sentence}"`, target = question.
Mirrors `backend/ml/generator/data/prepare_tydiqa.py`.

In [ ]:
import re
from datasets import load_dataset, Dataset

PREFIX = "buat pertanyaan: "
SENT = re.compile(r"(?<=[.!?])\s+")

def answer_sentence(ctx, ans, start):
    if start is None or start < 0 or ctx[start:start + len(ans)] != ans:
        start = max(ctx.find(ans), 0)
    end = start + len(ans)
    s = 0
    for m in SENT.finditer(ctx):
        if m.end() <= start: s = m.end()
        else: break
    e = len(ctx)
    for m in SENT.finditer(ctx):
        if m.start() >= end: e = m.start(); break
    return ctx[s:e].strip()

def build(split):
    rows = []
    for ex in split:
        if not str(ex["id"]).lower().startswith("indonesian"):
            continue
        ctx, q = ex["context"].strip(), ex["question"].strip()
        texts, starts = ex["answers"]["text"], ex["answers"]["answer_start"]
        if not (ctx and q and texts):
            continue
        ans = texts[0].strip()
        st = starts[0] if starts else ctx.find(ans)
        if not ans:
            continue
        rows.append({"input": PREFIX + answer_sentence(ctx, ans, st), "target": q})
    return rows

raw = load_dataset("tydiqa", "secondary_task")
train_rows, val_rows = build(raw["train"]), build(raw["validation"])
print("train:", len(train_rows), "| val:", len(val_rows))
print("sample:", train_rows[0])

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

## 3. Load tokenizer and base model

In [ ]:
from transformers import T5TokenizerFast, T5ForConditionalGeneration

MODEL = "Wikidepia/IndoT5-base"
tok = T5TokenizerFast.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)
print("params (M):", sum(p.numel() for p in model.parameters()) // 1_000_000)

## 4. Tokenize

In [ ]:
MAX_IN, MAX_OUT = 256, 64

def prep(batch):
    x = tok(batch["input"], max_length=MAX_IN, truncation=True)
    y = tok(text_target=batch["target"], max_length=MAX_OUT, truncation=True)
    x["labels"] = y["input_ids"]
    return x

train_tok = train_ds.map(prep, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(prep, batched=True, remove_columns=val_ds.column_names)

## 5. Train

`fp16=False` is the critical fix. ~30-60 min on a T4 for 4 epochs.

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)

args = Seq2SeqTrainingArguments(
    output_dir="indot5-quizgen",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    fp16=False,   # MUST stay False (T5 + fp16 -> NaN)
    bf16=False,   # T4 has no native bf16; fp32 is stable
    logging_steps=50,
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tok, model=model)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=collator, tokenizer=tok,
)
trainer.train()

## 6. Quick test

In [ ]:
def gen(text):
    ids = tok(PREFIX + text, return_tensors="pt", truncation=True,
              max_length=MAX_IN).input_ids.to(model.device)
    out = model.generate(ids, max_length=MAX_OUT, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

for t in [
    "Fotosintesis adalah proses pembentukan glukosa oleh tumbuhan hijau dengan bantuan cahaya matahari dan klorofil.",
    "Ibu kota Indonesia adalah Jakarta, yang terletak di pulau Jawa.",
]:
    print(t[:60], "->", gen(t))

## 7. Push to Hugging Face Hub

Paste a **WRITE** token from huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
REPO = "raviarnan/indot5-quizgen-asahlagi"
model.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("pushed to", REPO)

## 8. Use the fine-tuned model in the quiz-gen Space

In `huggingface/quizgen/app.py` (the Space repo), change the model id:

```python
# from:
MODEL = "Wikidepia/IndoT5-base"
# to:
MODEL = "raviarnan/indot5-quizgen-asahlagi"
```

Then push to the Space and trigger a **Factory rebuild**. Make sure the Space uses the
**same prefix** you trained with (`"buat pertanyaan: "`). The backend already points at the
Space via `HF_SPACE_URL`, so no backend change is needed.

**If quality is still weak**: raise `num_train_epochs` to 5-6, or switch to the answer-aware
(`<hl>`) input format (then the Space must wrap the chosen keyword in `<hl> ... <hl>` too).